In [1]:
import numpy as np
import jax.numpy as jnp
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

import hj_reachability as hj
from hj_reachability import sets
from hj_reachability.systems.doubleint import DoubleInt
from hj_reachability.zg_solver import ZGSolverSettings, step_until_converged

In [2]:
import os
# =========================
# Save path
# =========================

save_dir = Path("/home/zg0327/projects/HJR/hj_reachability/examples/Data/DoubleInt/clvf_qnorm_ell_only")
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / "doubleint_clvf_qnorm_ell_only_neuralop.npz"

print("save_path:", save_path)

save_path: /home/zg0327/projects/HJR/hj_reachability/examples/Data/DoubleInt/clvf_qnorm_ell_only/doubleint_clvf_qnorm_ell_only_neuralop.npz


In [30]:
# =========================
# Grid settings
# =========================

x1_min, x1_max = -2.0, 2.0
x2_min, x2_max = -2.0, 2.0
H, W = 101, 101

# =========================
# Dynamics settings
# =========================

u_max = 1.0
d1_max = 0.0
d2_max = 0.0

# =========================
# Solver settings
# =========================

initial_time = 0.0
target_time = -20.0

convergence_threshold = 1e-4
divergence_threshold = 20

# =========================
# Q-norm loss settings
# =========================

a_list = np.arange(0.1, 5.0 + 1e-9, 0.1).astype(np.float32)
b_list = np.array([1.0], dtype=np.float32)


gammas = np.array([0.3], dtype=np.float32)
# gammas = np.arange(0.1, 2.0 + 1e-9, 0.1).astype(np.float32)

target_radius = 0.0

print("a_list:", a_list)
print("b_list:", b_list)
print("gammas:", gammas)
print("num samples:", len(a_list) * len(b_list) * len(gammas))

a_list: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.  1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8
 1.9 2.  2.1 2.2 2.3 2.4 2.5 2.6 2.7 2.8 2.9 3.  3.1 3.2 3.3 3.4 3.5 3.6
 3.7 3.8 3.9 4.  4.1 4.2 4.3 4.4 4.5 4.6 4.7 4.8 4.9 5. ]
b_list: [1.]
gammas: [0.3]
num samples: 50


In [31]:
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    sets.Box(
        np.array([x1_min, x2_min]),
        np.array([x1_max, x2_max])
    ),
    (H, W)
)

x1s = np.asarray(grid.coordinate_vectors[0], dtype=np.float32)
x2s = np.asarray(grid.coordinate_vectors[1], dtype=np.float32)

X1, X2 = np.meshgrid(x1s, x2s, indexing="ij")

def make_ell_Q(a, b, target_radius=0.0):
    """
    ell_Q(x) = sqrt(a*x1^2 + b*x2^2) - target_radius,
    Q = diag(a,b).
    """
    ell = np.sqrt(a * X1**2 + b * X2**2) - target_radius
    return ell.astype(np.float32)

def compute_clvf_one(loss_values, gamma):
    """
    Compute one CLVF for the double integrator using CLVF_JAX.

    Parameters
    ----------
    loss_values : np.ndarray, shape [H, W]
        ell_Q(x) on the grid.
    gamma : float
        Exponential amplifier parameter.

    Returns
    -------
    V : np.ndarray, shape [H, W]
        Converged CLVF value function.
    """

    dynamics = DoubleInt(
        u_max=u_max,
        d1_max=d1_max,
        d2_max=d2_max,
        gamma=float(gamma),
        control_mode="min",
        disturbance_mode="max",
    )

    solver_settings = ZGSolverSettings(
        convergence_threshold=convergence_threshold,
        divergence_threshold=divergence_threshold,
        value_postprocessor = hj.solver.static_obstacle(loss_values),
    )

    values0 = jnp.asarray(loss_values)

    V = step_until_converged(
        solver_settings=solver_settings,
        dynamics=dynamics,
        grid=grid,
        time=initial_time,
        values=values0,
        target_time=target_time,
        convergence_threshold=convergence_threshold,
        progress_bar=True,
    )

    V = np.asarray(V, dtype=np.float32)

    return V

In [32]:
ells = []
Vs = []

gammas_all = []
a_all = []
b_all = []
Qs = []

sample_id = 0
total = len(a_list) * len(b_list) * len(gammas)

for a in tqdm(a_list, desc="a loop"):
    b = np.float32(1.0)  # fixed b

    ell = make_ell_Q(a, b, target_radius=target_radius)

    for gamma in gammas:
        sample_id += 1
        print(f"\nComputing sample {sample_id}/{total}: a={a}, b={b}, gamma={gamma}")

        V = compute_clvf_one(
            loss_values=ell,
            gamma=float(gamma),
        )

        ell_np = np.asarray(ell, dtype=np.float32)
        V_np = np.asarray(V, dtype=np.float32)

        if ell_np.shape != (H, W):
            raise ValueError(f"ell shape mismatch: got {ell_np.shape}, expected {(H, W)}")

        if V_np.shape != (H, W):
            raise ValueError(f"V shape mismatch: got {V_np.shape}, expected {(H, W)}")

        Q = np.array(
            [
                [a, 0.0],
                [0.0, b],
            ],
            dtype=np.float32
        )

        ells.append(ell_np)
        Vs.append(V_np)

        gammas_all.append(np.float32(gamma))
        a_all.append(np.float32(a))
        b_all.append(np.float32(b))
        Qs.append(Q)

ells = np.stack(ells, axis=0).astype(np.float32)              # [N, H, W]
Vs = np.stack(Vs, axis=0).astype(np.float32)                  # [N, H, W]

gammas_all = np.array(gammas_all, dtype=np.float32)           # [N]
a_all = np.array(a_all, dtype=np.float32)                     # [N]
b_all = np.array(b_all, dtype=np.float32)                     # [N]
Qs = np.stack(Qs, axis=0).astype(np.float32)                  # [N, 2, 2]

print("ells:", ells.shape)
print("Vs:", Vs.shape)
print("gammas_all:", gammas_all.shape)
print("a_all:", a_all.shape)
print("b_all:", b_all.shape)
print("Qs:", Qs.shape)

a loop:   0%|          | 0/50 [00:00<?, ?it/s]


Computing sample 1/50: a=0.10000000149011612, b=1.0, gamma=0.30000001192092896


t=-0.0100, max|ΔV|=6.911169e-03, max|ΔV|/dt=6.911169e-01
t=-0.0200, max|ΔV|=4.884822e-03, max|ΔV|/dt=4.884822e-01
t=-0.0300, max|ΔV|=3.438776e-03, max|ΔV|/dt=3.438776e-01
t=-0.0400, max|ΔV|=2.406619e-03, max|ΔV|/dt=2.406619e-01
t=-0.0500, max|ΔV|=1.840889e-03, max|ΔV|/dt=1.840890e-01
t=-0.0600, max|ΔV|=1.848936e-03, max|ΔV|/dt=1.848937e-01
t=-0.0700, max|ΔV|=1.852393e-03, max|ΔV|/dt=1.852394e-01
t=-0.0800, max|ΔV|=1.855791e-03, max|ΔV|/dt=1.855791e-01
t=-0.0900, max|ΔV|=1.861155e-03, max|ΔV|/dt=1.861155e-01
t=-0.1000, max|ΔV|=1.862466e-03, max|ΔV|/dt=1.862467e-01
t=-0.1100, max|ΔV|=1.859188e-03, max|ΔV|/dt=1.859189e-01
t=-0.1200, max|ΔV|=1.868844e-03, max|ΔV|/dt=1.868844e-01
t=-0.1300, max|ΔV|=1.874685e-03, max|ΔV|/dt=1.874686e-01
t=-0.1400, max|ΔV|=1.876593e-03, max|ΔV|/dt=1.876592e-01
t=-0.1500, max|ΔV|=1.875520e-03, max|ΔV|/dt=1.875519e-01
t=-0.1600, max|ΔV|=1.880765e-03, max|ΔV|/dt=1.880764e-01
t=-0.1700, max|ΔV|=1.889288e-03, max|ΔV|/dt=1.889287e-01
t=-0.1800, max|ΔV|=1.893580e-03

t=-4.1700, max|ΔV|=2.697706e-03, max|ΔV|/dt=2.697645e-01
t=-4.1800, max|ΔV|=2.656221e-03, max|ΔV|/dt=2.656161e-01
t=-4.1900, max|ΔV|=2.612352e-03, max|ΔV|/dt=2.612292e-01
t=-4.2000, max|ΔV|=2.566576e-03, max|ΔV|/dt=2.566517e-01
t=-4.2100, max|ΔV|=2.518892e-03, max|ΔV|/dt=2.518835e-01
t=-4.2200, max|ΔV|=2.470016e-03, max|ΔV|/dt=2.469960e-01
t=-4.2300, max|ΔV|=2.419949e-03, max|ΔV|/dt=2.419893e-01
t=-4.2400, max|ΔV|=2.369404e-03, max|ΔV|/dt=2.369350e-01
t=-4.2500, max|ΔV|=2.318144e-03, max|ΔV|/dt=2.318091e-01
t=-4.2600, max|ΔV|=2.267122e-03, max|ΔV|/dt=2.267070e-01
t=-4.2700, max|ΔV|=2.217054e-03, max|ΔV|/dt=2.217004e-01
t=-4.2800, max|ΔV|=2.167940e-03, max|ΔV|/dt=2.167891e-01
t=-4.2900, max|ΔV|=2.120495e-03, max|ΔV|/dt=2.120446e-01
t=-4.3000, max|ΔV|=2.075434e-03, max|ΔV|/dt=2.075386e-01
t=-4.3100, max|ΔV|=2.033234e-03, max|ΔV|/dt=2.033187e-01
t=-4.3200, max|ΔV|=1.994371e-03, max|ΔV|/dt=1.994326e-01
t=-4.3300, max|ΔV|=1.960278e-03, max|ΔV|/dt=1.960233e-01
t=-4.3400, max|ΔV|=1.930237e-03

t=-8.1801, max|ΔV|=4.198551e-04, max|ΔV|/dt=4.198455e-02
t=-8.1901, max|ΔV|=4.258156e-04, max|ΔV|/dt=4.258058e-02
t=-8.2001, max|ΔV|=4.303455e-04, max|ΔV|/dt=4.303357e-02
t=-8.2101, max|ΔV|=4.339218e-04, max|ΔV|/dt=4.339119e-02
t=-8.2201, max|ΔV|=4.408360e-04, max|ΔV|/dt=4.408259e-02
t=-8.2301, max|ΔV|=4.470348e-04, max|ΔV|/dt=4.470246e-02
t=-8.2401, max|ΔV|=4.520416e-04, max|ΔV|/dt=4.520313e-02
t=-8.2501, max|ΔV|=4.558563e-04, max|ΔV|/dt=4.558459e-02
t=-8.2601, max|ΔV|=4.627705e-04, max|ΔV|/dt=4.627599e-02
t=-8.2701, max|ΔV|=4.692078e-04, max|ΔV|/dt=4.691970e-02
t=-8.2801, max|ΔV|=4.744530e-04, max|ΔV|/dt=4.744421e-02
t=-8.2901, max|ΔV|=4.782677e-04, max|ΔV|/dt=4.782567e-02
t=-8.3001, max|ΔV|=4.851818e-04, max|ΔV|/dt=4.851707e-02
t=-8.3101, max|ΔV|=4.923344e-04, max|ΔV|/dt=4.923231e-02
t=-8.3201, max|ΔV|=4.978180e-04, max|ΔV|/dt=4.978066e-02
t=-8.3301, max|ΔV|=5.018711e-04, max|ΔV|/dt=5.018596e-02
t=-8.3401, max|ΔV|=5.090237e-04, max|ΔV|/dt=5.090120e-02
t=-8.3501, max|ΔV|=5.164146e-04

...

 19%|#8        |  3.8000/20.0 [00:00<00:00, 20.27sim_s/s]
a loop:  98%|█████████▊| 49/50 [00:31<00:00,  2.11it/s]

t=-0.0100, max|ΔV|=5.083609e-02, max|ΔV|/dt=5.083609e+00
t=-0.0200, max|ΔV|=5.097437e-02, max|ΔV|/dt=5.097437e+00
t=-0.0300, max|ΔV|=5.111361e-02, max|ΔV|/dt=5.111361e+00
t=-0.0400, max|ΔV|=5.125427e-02, max|ΔV|/dt=5.125427e+00
t=-0.0500, max|ΔV|=5.139542e-02, max|ΔV|/dt=5.139543e+00
t=-0.0600, max|ΔV|=5.153561e-02, max|ΔV|/dt=5.153562e+00
t=-0.0700, max|ΔV|=5.167580e-02, max|ΔV|/dt=5.167581e+00
t=-0.0800, max|ΔV|=5.181551e-02, max|ΔV|/dt=5.181552e+00
t=-0.0900, max|ΔV|=5.195427e-02, max|ΔV|/dt=5.195428e+00
t=-0.1000, max|ΔV|=5.209351e-02, max|ΔV|/dt=5.209352e+00
t=-0.1100, max|ΔV|=5.223179e-02, max|ΔV|/dt=5.223180e+00
t=-0.1200, max|ΔV|=5.236864e-02, max|ΔV|/dt=5.236865e+00
t=-0.1300, max|ΔV|=5.250549e-02, max|ΔV|/dt=5.250550e+00
t=-0.1400, max|ΔV|=5.264187e-02, max|ΔV|/dt=5.264184e+00
t=-0.1500, max|ΔV|=5.277729e-02, max|ΔV|/dt=5.277726e+00
t=-0.1600, max|ΔV|=5.291176e-02, max|ΔV|/dt=5.291173e+00
t=-0.1700, max|ΔV|=5.304527e-02, max|ΔV|/dt=5.304524e+00
t=-0.1800, max|ΔV|=5.317688e-02

t=-0.0100, max|ΔV|=5.147362e-02, max|ΔV|/dt=5.147362e+00
t=-0.0200, max|ΔV|=5.161190e-02, max|ΔV|/dt=5.161190e+00
t=-0.0300, max|ΔV|=5.175209e-02, max|ΔV|/dt=5.175209e+00
t=-0.0400, max|ΔV|=5.189323e-02, max|ΔV|/dt=5.189323e+00
t=-0.0500, max|ΔV|=5.203438e-02, max|ΔV|/dt=5.203439e+00
t=-0.0600, max|ΔV|=5.217552e-02, max|ΔV|/dt=5.217553e+00
t=-0.0700, max|ΔV|=5.231667e-02, max|ΔV|/dt=5.231668e+00
t=-0.0800, max|ΔV|=5.245638e-02, max|ΔV|/dt=5.245639e+00
t=-0.0900, max|ΔV|=5.259609e-02, max|ΔV|/dt=5.259610e+00
t=-0.1000, max|ΔV|=5.273485e-02, max|ΔV|/dt=5.273486e+00
t=-0.1100, max|ΔV|=5.287361e-02, max|ΔV|/dt=5.287362e+00
t=-0.1200, max|ΔV|=5.301189e-02, max|ΔV|/dt=5.301190e+00
t=-0.1300, max|ΔV|=5.314922e-02, max|ΔV|/dt=5.314923e+00
t=-0.1400, max|ΔV|=5.328560e-02, max|ΔV|/dt=5.328557e+00
t=-0.1500, max|ΔV|=5.342102e-02, max|ΔV|/dt=5.342099e+00
t=-0.1600, max|ΔV|=5.355597e-02, max|ΔV|/dt=5.355594e+00
t=-0.1700, max|ΔV|=5.368996e-02, max|ΔV|/dt=5.368993e+00
t=-0.1800, max|ΔV|=5.382252e-02

 21%|##1       |  4.2300/20.0 [00:00<00:00, 19.11sim_s/s]
a loop: 100%|██████████| 50/50 [00:32<00:00,  1.54it/s]

t=-3.8900, max|ΔV|=1.144409e-05, max|ΔV|/dt=1.144410e-03
t=-3.9000, max|ΔV|=1.144409e-05, max|ΔV|/dt=1.144410e-03
t=-3.9100, max|ΔV|=1.144409e-05, max|ΔV|/dt=1.144410e-03
t=-3.9200, max|ΔV|=1.525879e-05, max|ΔV|/dt=1.525880e-03
t=-3.9300, max|ΔV|=1.525879e-05, max|ΔV|/dt=1.525880e-03
t=-3.9400, max|ΔV|=1.716614e-05, max|ΔV|/dt=1.716615e-03
t=-3.9500, max|ΔV|=1.907349e-05, max|ΔV|/dt=1.907350e-03
t=-3.9600, max|ΔV|=1.907349e-05, max|ΔV|/dt=1.907350e-03
t=-3.9700, max|ΔV|=1.716614e-05, max|ΔV|/dt=1.716615e-03
t=-3.9800, max|ΔV|=1.525879e-05, max|ΔV|/dt=1.525880e-03
t=-3.9900, max|ΔV|=1.525879e-05, max|ΔV|/dt=1.525880e-03
t=-4.0000, max|ΔV|=1.525879e-05, max|ΔV|/dt=1.525880e-03
t=-4.0100, max|ΔV|=1.335144e-05, max|ΔV|/dt=1.335114e-03
t=-4.0200, max|ΔV|=1.144409e-05, max|ΔV|/dt=1.144383e-03
t=-4.0300, max|ΔV|=7.629395e-06, max|ΔV|/dt=7.629220e-04
t=-4.0400, max|ΔV|=7.629395e-06, max|ΔV|/dt=7.629220e-04
t=-4.0500, max|ΔV|=5.722046e-06, max|ΔV|/dt=5.721915e-04
t=-4.0600, max|ΔV|=3.814697e-06

In [33]:
x_data = ells[:, None, :, :]   # [N, 1, H, W]
y_data = Vs[:, None, :, :]     # [N, 1, H, W]

print("x_data:", x_data.shape)
print("y_data:", y_data.shape)
print("x_data dtype:", x_data.dtype)
print("y_data dtype:", y_data.dtype)

x_data: (50, 1, 101, 101)
y_data: (50, 1, 101, 101)
x_data dtype: float32
y_data dtype: float32


In [34]:
np.savez_compressed(
    save_path,

    # Neuralop-ready tensors as numpy arrays
    x_data=x_data,              # [N, 1, H, W], input = ell_Q only
    y_data=y_data,              # [N, 1, H, W], output = V_gamma

    # Raw data
    ells=ells,                  # [N, H, W]
    Vs=Vs,                      # [N, H, W]

    # Metadata
    gammas=gammas_all,          # [N]
    a_values=a_all,             # [N]
    b_values=b_all,             # [N]
    Qs=Qs,                      # [N, 2, 2]

    # Grid
    x1s=x1s,                    # [H]
    x2s=x2s,                    # [W]

    # Experiment settings
    u_max=np.float32(u_max),
    d1_max=np.float32(d1_max),
    d2_max=np.float32(d2_max),
    target_radius=np.float32(target_radius),
    initial_time=np.float32(initial_time),
    target_time=np.float32(target_time),
    convergence_threshold=np.float32(convergence_threshold),
    divergence_threshold=np.float32(divergence_threshold),

    input_channels=np.array(["ell_Q"]),
    output_channels=np.array(["V_gamma"]),
    system_name=np.array("DoubleInt"),
)

print("Saved to:")
print(save_path)

Saved to:
/home/zg0327/projects/HJR/hj_reachability/examples/Data/DoubleInt/clvf_qnorm_ell_only/doubleint_clvf_qnorm_ell_only_neuralop.npz


In [35]:
data = np.load(save_path, allow_pickle=True)

print("x_data:", data["x_data"].shape)
print("y_data:", data["y_data"].shape)
print("gammas:", data["gammas"].shape)
print("Qs:", data["Qs"].shape)
print("x1s:", data["x1s"].shape)
print("x2s:", data["x2s"].shape)

print("input_channels:", data["input_channels"])
print("output_channels:", data["output_channels"])
print("system_name:", data["system_name"])

x_data: (50, 1, 101, 101)
y_data: (50, 1, 101, 101)
gammas: (50,)
Qs: (50, 2, 2)
x1s: (101,)
x2s: (101,)
input_channels: ['ell_Q']
output_channels: ['V_gamma']
system_name: DoubleInt
